In [1]:
# =====================================================================================
# Project: PTB-XL Metadata Audit  -  read-only integrity and metadata audit
# Code version: v0.1.2   (extends v0.1.1)
# Date: 2026-09-10
# Purpose: provenance, data shape, recording metadata, demographics, split integrity,
#          metadata/patient entanglement, concentration, label derivation, crosstabs,
#          temporal confounding, and a permutation chance baseline for metadata-label
#          association.
# Independent variable: none - this version trains no model and computes no
#          performance metric.
#
# Changes vs v0.1.1:
#   1) Robust SHA256SUMS lookup: tries SHA256SUMS / SHA256SUMS.txt / sha256sums /
#      sha256sums.txt in that order; if none exist, lists files containing "sha"
#      (case-insensitive) in the dataset root instead of silently failing.
#   2) RECORDS mismatch is diagnosed, not just flagged: detects the case where a
#      missing newline concatenates a records100/... path directly with a
#      records500/... path, and downgrades the message from a warning to an
#      explanatory note when the split reproduces two known CSV paths.
#   3) New Section 5.1 - metadata/patient entanglement (are device/site/nurse
#      patient-level or record-level attributes?).
#   4) New Section 5.2 - concentration / effective cardinality (top-k share,
#      Shannon entropy, effective number of categories) for device/site/nurse.
#   5) New Section 7.6 - temporal confounding: recording_year x strat_fold and
#      device x recording_year crosstabs with Cramer's V.
#   6) New Section 7.7 - permutation chance baseline for metadata<->label
#      association (folds 1-9 only): observed bias-corrected Cramer's V and
#      max_abs_deviation_pp against a null built by shuffling the label vector.
#   7) Likelihood-sensitivity block now also reports per-superclass counts under
#      the alternative (likelihood > 0) convention.
#   8) The console no longer dumps the full manifest; it prints only the
#      top-level keys with each value's serialized size. The file on disk is
#      still complete.
#
# Allowed inputs (read-only): ptbxl_database.csv | scp_statements.csv | RECORDS |
#                              a SHA256SUMS-like file | ptbxl_v102_changelog.txt |
#                              ptbxl_v103_changelog.txt
# No signal file under records100/ or records500/ is ever opened or hashed.
# wfdb is not imported.
#
# Outputs (all in OUT_DIR):
#    1)  v0.1.2_column_inventory.csv
#    2)  v0.1.2_labels.csv
#    3)  v0.1.2_crosstab_device_fold.csv                       + _rowshare_raw
#    4)  v0.1.2_crosstab_site_fold.csv                         + _rowshare_raw
#    5)  v0.1.2_crosstab_nurse_fold.csv                        + _rowshare_raw
#    6)  v0.1.2_prevalence_device_superclass_folds1to9.csv     + _raw
#    7)  v0.1.2_prevalence_site_superclass_folds1to9.csv       + _raw
#    8)  v0.1.2_crosstab_recording_year_fold.csv
#    9)  v0.1.2_crosstab_device_recording_year.csv
#   10)  v0.1.2_assoc_permutation_device.csv
#   11)  v0.1.2_assoc_permutation_site.csv
#   12)  v0.1.2_assoc_permutation_nurse.csv
#   13)  v0.1.2_manifest.json
#
# Prohibited: training any model | computing AUROC/accuracy/any performance
#             metric | plotting anything | writing or modifying any file inside
#             the dataset folder.
# =====================================================================================

import os, sys, json, ast, hashlib, platform
from datetime import datetime
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

# ------------------------------- configuration ---------------------------------------
DATA_ROOT = r"D:\ptbxl-metadata-audit\data\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
OUT_DIR   = r"D:\ptbxl-metadata-audit\outputs\v0.1.2"
CODE_VERSION = "v0.1.2"
PREFIX = CODE_VERSION + "_"
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

N_PERM = 1000
PERM_SEED = 42

# Published anchor numbers (for comparison only, never enforced) - source: the
# original PTB-XL paper.
ANCHORS = {"n_records": 21799, "n_patients": 18869,
           "NORM": 9514, "MI": 5469, "STTC": 5235, "CD": 4898, "HYP": 2649}

os.makedirs(OUT_DIR, exist_ok=True)

# Safety guard: the output directory must never live inside the dataset folder.
_root_abs, _out_abs = os.path.abspath(DATA_ROOT), os.path.abspath(OUT_DIR)
assert not _out_abs.lower().startswith(_root_abs.lower()), "OUT_DIR is inside the dataset folder - aborting."

RUN_STARTED = datetime.now().astimezone().isoformat()
OUTPUT_FILES = []          # names of files written
WARNINGS = []              # warnings collected for the final summary
LEAKAGE_FAIL = False


def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def save_csv(obj, name, index=True):
    p = os.path.join(OUT_DIR, name)
    obj.to_csv(p, index=index, encoding="utf-8-sig")
    OUTPUT_FILES.append(name)
    print("[saved] " + p)
    return p


def hdr(t):
    print("\n" + "=" * 100)
    print(t)
    print("=" * 100)


def warn(msg):
    WARNINGS.append(msg)
    print("[WARN] " + msg)


def note(msg):
    print("[NOTE] " + msg)


def _json_default(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


def cramers_v_arr(obs):
    """Raw and Bergsma bias-corrected Cramer's V from a 2D count matrix (numpy array)."""
    obs = np.asarray(obs, dtype=float)
    n = obs.sum()
    if n <= 0:
        return {"n": 0, "shape": list(obs.shape), "chi2": float("nan"), "dof": 0,
                "cramers_v": float("nan"), "cramers_v_biascorrected": float("nan")}
    row = obs.sum(axis=1, keepdims=True)
    col = obs.sum(axis=0, keepdims=True)
    exp = row @ col / n
    with np.errstate(divide="ignore", invalid="ignore"):
        chi2 = float(np.nansum(np.where(exp > 0, (obs - exp) ** 2 / np.where(exp > 0, exp, 1.0), 0.0)))
    r, k = obs.shape
    dof = (r - 1) * (k - 1)
    v = float(np.sqrt(chi2 / (n * max(min(r, k) - 1, 1))))
    denom_n1 = max(n - 1, 1e-12)
    phi2 = chi2 / n
    phi2c = max(0.0, phi2 - dof / denom_n1)
    rc = r - (r - 1) ** 2 / denom_n1
    kc = k - (k - 1) ** 2 / denom_n1
    denom = max(min(rc, kc) - 1, 1e-12)
    vc = float(np.sqrt(max(phi2c, 0.0) / denom))
    return {"n": int(n), "shape": [r, k], "chi2": chi2, "dof": dof,
            "cramers_v": v, "cramers_v_biascorrected": vc}


def cramers_v(ct):
    """Cramer's V for a pandas crosstab (thin wrapper around cramers_v_arr)."""
    return cramers_v_arr(ct.values)


# =====================================================================================
# SECTION 1 - PROVENANCE
# =====================================================================================
hdr("SECTION 1 - PROVENANCE")

print("run started         :", RUN_STARTED)
print("code version        :", CODE_VERSION)
print("python              :", sys.version.replace("\n", " "))
print("platform            :", platform.platform())
print("machine / processor :", platform.machine(), "|", platform.processor())
print("numpy               :", np.__version__)
print("pandas              :", pd.__version__)
print("dataset folder name :", os.path.basename(os.path.normpath(DATA_ROOT)))
print("dataset root        :", DATA_ROOT)
print("output dir          :", OUT_DIR)

DB_CSV      = os.path.join(DATA_ROOT, "ptbxl_database.csv")
SCP_CSV     = os.path.join(DATA_ROOT, "scp_statements.csv")
RECORDS_TXT = os.path.join(DATA_ROOT, "RECORDS")
CHANGELOGS  = ["ptbxl_v102_changelog.txt", "ptbxl_v103_changelog.txt"]

for p in (DB_CSV, SCP_CSV, RECORDS_TXT):
    assert os.path.isfile(p), "file not found: " + p

print("\n--- SHA-256 of the main input files (computed) ---")
INPUT_HASHES = {}
for p in (DB_CSV, SCP_CSV):
    h = sha256_of(p)
    INPUT_HASHES[os.path.basename(p)] = h
    print("{:26s} {}   ({:,} bytes)".format(os.path.basename(p), h, os.path.getsize(p)))

# --- Robust SHA256SUMS lookup (fix vs v0.1.1) ---
print("\n--- looking for a checksum file ---")
SUMS_CANDIDATES = ["SHA256SUMS", "SHA256SUMS.txt", "sha256sums", "sha256sums.txt"]
SUMS_TXT = None
SUMS_NAME_USED = None
for _name in SUMS_CANDIDATES:
    _p = os.path.join(DATA_ROOT, _name)
    if os.path.isfile(_p):
        SUMS_TXT = _p
        SUMS_NAME_USED = _name
        break

TARGETS = ["ptbxl_database.csv", "scp_statements.csv"]
SHA256SUMS_VERIFICATION = {}

if SUMS_TXT is not None:
    print("checksum file found: " + SUMS_NAME_USED)
    listed = {}
    n_lines = 0
    with open(SUMS_TXT, "r", encoding="utf-8", errors="replace") as f:
        for ln in f:
            ln = ln.strip()
            if not ln:
                continue
            n_lines += 1
            parts = ln.split(None, 1)
            if len(parts) != 2:
                continue
            hval = parts[0].strip().lower()
            fname = parts[1].strip().lstrip("*").replace("\\", "/")
            base = os.path.basename(fname)
            if base in TARGETS:
                listed[base] = hval
    print("lines in checksum file:", n_lines, "| target rows found:", len(listed))
    for t in TARGETS:
        computed = INPUT_HASHES[t].lower()
        if t not in listed:
            status, expected = "NOT_LISTED", None
        elif listed[t] == computed:
            status, expected = "MATCH", listed[t]
        else:
            status, expected = "MISMATCH", listed[t]
        SHA256SUMS_VERIFICATION[t] = {"status": status, "expected": expected, "computed": computed,
                                       "source_file": SUMS_NAME_USED}
        print("  {:26s} -> {}".format(t, status))
        if status != "MATCH":
            print("      expected: {}".format(expected))
            print("      computed: {}".format(computed))
        if status == "MISMATCH":
            warn("{}'s hash does not match the checksum file - the file is corrupted or "
                 "incomplete. No downstream number is trustworthy until this is resolved.".format(t))
        if status == "NOT_LISTED":
            warn("{} is not listed in the checksum file - its integrity cannot be verified.".format(t))
else:
    print("no checksum file found among candidates:", SUMS_CANDIDATES)
    sha_like = sorted([f for f in os.listdir(DATA_ROOT) if "sha" in f.lower()])
    print("files in dataset root containing 'sha' (case-insensitive):",
          sha_like if sha_like else "(none)")
    SHA256SUMS_VERIFICATION = {"_file": "NOT_FOUND", "candidates_tried": SUMS_CANDIDATES,
                                "sha_like_files_found": sha_like}
    warn("No checksum file found under any of {} - integrity verification skipped.".format(SUMS_CANDIDATES))

print("\n--- SHA-256 of other inputs read ---")
OTHER_HASHES = {}
for fn in [os.path.basename(RECORDS_TXT)] + CHANGELOGS:
    p = os.path.join(DATA_ROOT, fn)
    if os.path.isfile(p):
        h = sha256_of(p)
        OTHER_HASHES[fn] = h
        print("{:26s} {}   ({:,} bytes)".format(fn, h, os.path.getsize(p)))
    else:
        warn("not found: " + p)

for fn in CHANGELOGS:
    p = os.path.join(DATA_ROOT, fn)
    print("\n" + "-" * 100)
    print("--- full content of " + fn + " ---")
    print("-" * 100)
    if os.path.isfile(p):
        with open(p, "r", encoding="utf-8", errors="replace") as f:
            print(f.read())
    else:
        print("[WARN] file does not exist.")


# =====================================================================================
# SECTION 2 - DATA SHAPE
# =====================================================================================
hdr("SECTION 2 - DATA SHAPE")

df = pd.read_csv(DB_CSV, index_col="ecg_id")

print("shape (rows, cols)      :", df.shape)
print("unique ecg_id           :", df.index.nunique())
print("duplicated ecg_id       :", int(df.index.duplicated().sum()))
print("unique patient_id       :", df["patient_id"].nunique())
print("anchor n_records/n_pat  :", ANCHORS["n_records"], "/", ANCHORS["n_patients"])
if len(df) != ANCHORS["n_records"]:
    warn("record count ({}) does not match the published anchor ({}).".format(len(df), ANCHORS["n_records"]))
if df["patient_id"].nunique() != ANCHORS["n_patients"]:
    warn("patient count ({}) does not match the published anchor ({}).".format(
        df["patient_id"].nunique(), ANCHORS["n_patients"]))

print("\n--- columns, dtype, missingness, cardinality ---")
inventory = pd.DataFrame({
    "column":      list(df.columns),
    "dtype":       [str(t) for t in df.dtypes],
    "n_missing":   df.isna().sum().values,
    "pct_missing": np.round(df.isna().mean().values * 100, 3),
    "n_unique":    [int(df[c].nunique(dropna=True)) for c in df.columns],
})
print(inventory.to_string(index=False))
save_csv(inventory, PREFIX + "column_inventory.csv", index=False)

# --- RECORDS consistency check + diagnosis of the concatenated-line mismatch ---
print("\n--- RECORDS consistency check (reading the path list only) ---")
with open(RECORDS_TXT, "r", encoding="utf-8", errors="replace") as f:
    records_lines = [ln.strip() for ln in f if ln.strip()]
rec_set = set(records_lines)
csv_paths = set(df["filename_lr"].astype(str)) | set(df["filename_hr"].astype(str))
print("RECORDS lines                 :", len(records_lines), "| unique:", len(rec_set))
print("unique paths in CSV           :", len(csv_paths))
only_csv = sorted(csv_paths - rec_set)
only_rec = sorted(rec_set - csv_paths)
print("in CSV but not in RECORDS     :", len(only_csv), ("| sample: " + str(only_csv[:5])) if only_csv else "")
print("in RECORDS but not in CSV     :", len(only_rec), ("| sample: " + str(only_rec[:5])) if only_rec else "")

RECORDS_MISMATCH_DIAGNOSIS = {"explained_lines": [], "unexplained_lines": []}
if only_rec:
    print("\n--- diagnosing the RECORDS mismatch ---")
    for line in only_rec:
        found = None
        for marker in ("records500/", "records100/"):
            idx = line.find(marker, 1)
            if idx > 0:
                part1, part2 = line[:idx], line[idx:]
                if part1 in csv_paths and part2 in csv_paths:
                    found = (part1, part2)
                    break
        if found is not None:
            RECORDS_MISMATCH_DIAGNOSIS["explained_lines"].append(
                {"raw_line": line, "split_into": list(found)})
        else:
            RECORDS_MISMATCH_DIAGNOSIS["unexplained_lines"].append(line)

    n_explained = len(RECORDS_MISMATCH_DIAGNOSIS["explained_lines"])
    n_unexplained = len(RECORDS_MISMATCH_DIAGNOSIS["unexplained_lines"])
    print("explained by a missing newline :", n_explained)
    for item in RECORDS_MISMATCH_DIAGNOSIS["explained_lines"]:
        print("  {} -> {}".format(item["raw_line"], item["split_into"]))
    print("still unexplained              :", n_unexplained)
    if RECORDS_MISMATCH_DIAGNOSIS["unexplained_lines"]:
        print("  sample:", RECORDS_MISMATCH_DIAGNOSIS["unexplained_lines"][:5])

    if n_explained > 0 and n_unexplained == 0:
        note("All RECORDS/CSV mismatches are explained by a missing newline in the RECORDS "
             "file itself, which concatenates a records100/... path directly with a "
             "records500/... path on one line. This is a benign artifact of the dataset "
             "file, not a data-integrity problem.")
    elif n_explained > 0:
        warn("{} of {} RECORDS mismatches are explained by a missing-newline concatenation; "
             "{} remain unexplained and need manual review.".format(
                 n_explained, n_explained + n_unexplained, n_unexplained))
    else:
        warn("RECORDS does not match the CSV paths and none of the mismatches fit the "
             "known missing-newline pattern - needs manual review.")
elif only_csv:
    warn("CSV has paths not present in RECORDS, but RECORDS has no orphan lines to diagnose "
         "- check whether RECORDS is simply incomplete.")


# =====================================================================================
# SECTION 3 - RECORDING METADATA
# =====================================================================================
hdr("SECTION 3 - RECORDING METADATA (device / site / nurse)")

for col in ["device", "site", "nurse"]:
    print("\n--- " + col + " ---")
    print("dtype:", df[col].dtype,
          "| nunique (non-null):", int(df[col].nunique(dropna=True)),
          "| missing:", int(df[col].isna().sum()))
    vc = df[col].value_counts(dropna=False).to_frame("n")
    vc["pct"] = np.round(vc["n"] / len(df) * 100, 3)
    print(vc.to_string())

print("\n--- recording_date ---")
rd = pd.to_datetime(df["recording_date"], errors="coerce")
n_unparsable = int(rd.isna().sum() - df["recording_date"].isna().sum())
print("min:", rd.min(), "| max:", rd.max())
print("missing:", int(df["recording_date"].isna().sum()), "| unparsable:", n_unparsable)
print("\nyearly distribution:")
print(rd.dt.year.value_counts(dropna=False).sort_index().to_frame("n").to_string())

recording_year_num = rd.dt.year  # kept for Section 7.6


# =====================================================================================
# SECTION 4 - DEMOGRAPHICS
# =====================================================================================
hdr("SECTION 4 - DEMOGRAPHICS")

print("--- sex ---")
vc = df["sex"].value_counts(dropna=False).to_frame("n")
vc["pct"] = np.round(vc["n"] / len(df) * 100, 3)
print(vc.to_string())

print("\n--- age ---")
age = df["age"]
print("dtype:", age.dtype, "| missing:", int(age.isna().sum()))
print("min :", age.min(), "| max:", age.max(), "| mean:", round(float(age.mean()), 3))
for q in [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]:
    print("  p{:g} = {}".format(q * 100, age.quantile(q)))

print("\ntop 15 most frequent age values (to spot sentinels):")
print(age.value_counts(dropna=False).head(15).to_frame("n").to_string())

print("\ncounts of suspicious / sentinel values:")
checks = [("age > 89", (age > 89)), ("age > 90", (age > 90)),
          ("age >= 100", (age >= 100)), ("age >= 120 (physiologically impossible)", (age >= 120)),
          ("age == 300", (age == 300)), ("age <= 0", (age <= 0))]
for name, mask in checks:
    print("  {:42s} : {}".format(name, int(mask.sum())))
print("  {:42s} : {}".format("age is non-integer", int((age.dropna() % 1 != 0).sum())))

n_sentinel = int((age >= 120).sum())
if n_sentinel > 0:
    warn("sentinel value found in age ({} records with age>=120). These are censored "
         "ages and must be handled explicitly before any modeling.".format(n_sentinel))

print("\nfull distribution of ages >= 80 (the censoring boundary is usually here):")
print(age[age >= 80].value_counts().sort_index().to_frame("n").to_string())

print("\n--- height / weight ---")
for c in ["height", "weight"]:
    print("\n{}: missing {} ({:.2f}%)".format(c, int(df[c].isna().sum()), df[c].isna().mean() * 100))
    print(df[c].describe().to_string())


# =====================================================================================
# SECTION 5 - SPLIT INTEGRITY
# =====================================================================================
hdr("SECTION 5 - SPLIT INTEGRITY")

print("--- strat_fold ---")
fold_tab = df["strat_fold"].value_counts(dropna=False).sort_index().to_frame("n_records")
fold_tab["pct"] = np.round(fold_tab["n_records"] / len(df) * 100, 3)
fold_tab["n_patients"] = df.groupby("strat_fold")["patient_id"].nunique()
print(fold_tab.to_string())

FOLD_TAB_JSON = {}
for k, row in fold_tab.iterrows():
    key = str(int(k)) if pd.notna(k) else "NaN"
    FOLD_TAB_JSON[key] = {
        "n_records": int(row["n_records"]),
        "pct": float(row["pct"]),
        "n_patients": int(row["n_patients"]) if pd.notna(row["n_patients"]) else None,
    }

print("\n--- patient-disjoint fold check ---")
folds_per_patient = df.groupby("patient_id")["strat_fold"].nunique()
violators = folds_per_patient[folds_per_patient > 1]
LEAKAGE_FAIL = len(violators) > 0
print("patients appearing in more than one fold:", len(violators))
if LEAKAGE_FAIL:
    warn("patient-level leakage in the official split - this is a blocker.")
    print("\nfirst 10 offending patients and their folds:")
    for pid in list(violators.index[:10]):
        sub = df[df["patient_id"] == pid]
        print("  patient_id={} | folds={} | n_records={}".format(
            pid, sorted(sub["strat_fold"].unique().tolist()), len(sub)))
else:
    print("PASS - no patient is shared between two folds.")

print("\n--- distribution of records per patient ---")
rec_per_patient = df.groupby("patient_id").size()
print(rec_per_patient.value_counts().sort_index().to_frame("n_patients").to_string())
print("max records per patient:", int(rec_per_patient.max()),
      "| mean:", round(float(rec_per_patient.mean()), 4))

if "validated_by_human" in df.columns:
    print("\n--- validated_by_human x strat_fold ---")
    ct_val = pd.crosstab(df["strat_fold"], df["validated_by_human"].fillna("__MISSING__"))
    print(ct_val.to_string())

# --- Noisy leakage banner (so it can't get lost in a long console) ---
if LEAKAGE_FAIL:
    print()
    for _ in range(5):
        print("!" * 100)
    print("!!!  LEAKAGE FAIL - patient-level leakage in the official split  !!!")
    print("!!!  {} patients appear in more than one strat_fold.".format(len(violators)))
    print("!!!  No performance number on this split is valid. Execution continues so that")
    print("!!!  the audit outputs are produced, but an AssertionError is raised at the end")
    print("!!!  of the cell.")
    for _ in range(5):
        print("!" * 100)
    print()


# =====================================================================================
# SECTION 5.1 - METADATA / PATIENT ENTANGLEMENT
# =====================================================================================
hdr("SECTION 5.1 - METADATA / PATIENT ENTANGLEMENT")

METADATA_PATIENT_ENTANGLEMENT = {}
for col in ["device", "site", "nurse"]:
    nunique_per_patient = df.groupby("patient_id")[col].nunique(dropna=True)
    n_patients_total = int(nunique_per_patient.shape[0])
    n_multi = int((nunique_per_patient > 1).sum())
    pct_multi = round(n_multi / n_patients_total * 100, 4) if n_patients_total else float("nan")
    dist = nunique_per_patient.value_counts().sort_index()

    print("\n--- " + col + " ---")
    print("patients with a single distinct value   :", n_patients_total - n_multi)
    print("patients with more than one distinct value:", n_multi,
          "({:.3f}% of {} patients)".format(pct_multi, n_patients_total))
    print("distribution of distinct-values-per-patient:")
    print(dist.to_frame("n_patients").to_string())

    METADATA_PATIENT_ENTANGLEMENT[col] = {
        "n_patients_total": n_patients_total,
        "n_patients_with_multiple_values": n_multi,
        "pct_patients_with_multiple_values": pct_multi,
        "distinct_values_per_patient_distribution": {str(int(k)): int(v) for k, v in dist.items()},
    }

    if n_multi == 0:
        note("{} is effectively a patient-level attribute (constant within every patient) "
             "on this dataset.".format(col))


# =====================================================================================
# SECTION 5.2 - CONCENTRATION / EFFECTIVE CARDINALITY
# =====================================================================================
hdr("SECTION 5.2 - CONCENTRATION / EFFECTIVE CARDINALITY")

METADATA_CONCENTRATION = {}
for col in ["device", "site", "nurse"]:
    vc = df[col].value_counts(dropna=True).sort_values(ascending=False)
    n_cat_nominal = int(df[col].nunique(dropna=True))
    total = int(vc.sum())
    shares = (vc / total).to_numpy()

    top1 = float(shares[:1].sum() * 100)
    top3 = float(shares[:3].sum() * 100)
    top5 = float(shares[:5].sum() * 100)
    entropy_bits = float(-(shares * np.log2(shares)).sum())
    entropy_nats = float(-(shares * np.log(shares)).sum())
    effective_n = float(np.exp(entropy_nats))

    print("\n--- " + col + " ---")
    print("nominal categories       :", n_cat_nominal)
    print("share of top 1 category  : {:.2f}%".format(top1))
    print("share of top 3 categories: {:.2f}%".format(top3))
    print("share of top 5 categories: {:.2f}%".format(top5))
    print("Shannon entropy          : {:.4f} bits".format(entropy_bits))
    print("effective n categories   : {:.2f}  (nominal = {})".format(effective_n, n_cat_nominal))

    METADATA_CONCENTRATION[col] = {
        "n_categories_nominal": n_cat_nominal,
        "top1_share_pct": round(top1, 3),
        "top3_share_pct": round(top3, 3),
        "top5_share_pct": round(top5, 3),
        "shannon_entropy_bits": round(entropy_bits, 4),
        "effective_n_categories": round(effective_n, 3),
    }
    if effective_n < n_cat_nominal * 0.5:
        warn("{}: effective cardinality ({:.1f}) is far below the nominal category count "
             "({}) - the column is much less informative than its cardinality suggests.".format(
                 col, effective_n, n_cat_nominal))


# =====================================================================================
# SECTION 6 - LABEL DERIVATION
# =====================================================================================
hdr("SECTION 6 - LABEL DERIVATION")

scp = pd.read_csv(SCP_CSV, index_col=0)
print("scp_statements shape:", scp.shape)
print("columns:", list(scp.columns))

diag = scp[scp["diagnostic"].fillna(0) == 1]
print("\nstatements with diagnostic == 1 :", len(diag), "of", len(scp))
print("\ndiagnostic_class distribution among them:")
print(diag["diagnostic_class"].value_counts(dropna=False).to_frame("n_statements").to_string())

code2class = {k: v for k, v in diag["diagnostic_class"].items() if isinstance(v, str)}

parsed = df["scp_codes"].apply(ast.literal_eval)
all_codes = set()
for d in parsed:
    all_codes.update(d.keys())
unknown_codes = sorted(all_codes - set(scp.index))
print("\nunique codes in scp_codes :", len(all_codes),
      "| codes not present in scp_statements:", len(unknown_codes))
if unknown_codes:
    warn("unknown codes in scp_codes: " + str(unknown_codes[:20]))

lik_counter = {}
for d in parsed:
    for v in d.values():
        lik_counter[v] = lik_counter.get(v, 0) + 1
print("\ndistribution of likelihood values in scp_codes (over all record-code pairs):")
print(pd.Series(lik_counter).sort_index().to_frame("n").to_string())

extra_classes = sorted({v for v in code2class.values()} - set(SUPERCLASSES))
if extra_classes:
    warn("diagnostic_class values outside the five superclasses: " + str(extra_classes))


def to_classes(d, min_likelihood=None):
    out = set()
    for k, v in d.items():
        if k not in code2class:
            continue
        if min_likelihood is not None and not (float(v) > min_likelihood):
            continue
        out.add(code2class[k])
    return out


# Convention used for the label file: keep all keys, no likelihood threshold
# (the benchmark convention).
LABEL_CONVENTION = "all keys of scp_codes mapped via diagnostic==1 (no likelihood threshold)"
rec_classes = parsed.apply(lambda d: to_classes(d, None))
lab = pd.DataFrame({c: rec_classes.apply(lambda s, c=c: int(c in s)) for c in SUPERCLASSES},
                   index=df.index)

print("\n--- record counts per superclass (convention: " + LABEL_CONVENTION + ") ---")
cmp_tab = pd.DataFrame({
    "derived": [int(lab[c].sum()) for c in SUPERCLASSES],
    "anchor_published": [ANCHORS[c] for c in SUPERCLASSES],
})
cmp_tab.index = SUPERCLASSES
cmp_tab["diff"] = cmp_tab["derived"] - cmp_tab["anchor_published"]
cmp_tab["prevalence_pct"] = np.round(cmp_tab["derived"] / len(df) * 100, 3)
print(cmp_tab.to_string())
if (cmp_tab["diff"] != 0).any():
    warn("derived counts differ from the published anchors - likely cause: dataset "
         "version drift or a convention difference. Cite the Section 1 changelogs in the report.")

n_labels = lab.sum(axis=1)
print("\nrecords with no superclass :", int((n_labels == 0).sum()),
      "({:.2f}%)".format((n_labels == 0).mean() * 100))
print("\ndistribution of labels per record:")
nl = n_labels.value_counts().sort_index().to_frame("n_records")
nl["pct"] = np.round(nl["n_records"] / len(df) * 100, 3)
print(nl.to_string())

alt = parsed.apply(lambda d: to_classes(d, 0.0))
n_changed = int((rec_classes != alt).sum())
lab_alt = pd.DataFrame({c: alt.apply(lambda s, c=c: int(c in s)) for c in SUPERCLASSES}, index=df.index)
alt_counts = {c: int(lab_alt[c].sum()) for c in SUPERCLASSES}

print("\nsensitivity: if only codes with likelihood>0 are kept,",
      n_changed, "records ({:.2f}%) change their label set.".format(n_changed / len(df) * 100))
print("\nper-superclass counts, primary vs alternative (likelihood>0) convention:")
lik_cmp = pd.DataFrame({
    "primary_count": [int(lab[c].sum()) for c in SUPERCLASSES],
    "alt_count": [alt_counts[c] for c in SUPERCLASSES],
}, index=SUPERCLASSES)
lik_cmp["diff"] = lik_cmp["alt_count"] - lik_cmp["primary_count"]
print(lik_cmp.to_string())

labels_out = pd.concat([df[["patient_id", "strat_fold"]], lab], axis=1).reset_index()
save_csv(labels_out, PREFIX + "labels.csv", index=False)


# =====================================================================================
# SECTION 7 - CROSSTABS  (the most important section)
# =====================================================================================
hdr("SECTION 7 - CROSSTABS (device / site / nurse  x  fold and superclass)")


def to_sortable_label(s, name, missing="__MISSING__"):
    """Zero-padded string label for numeric columns (e.g. site_007) so lexical
    sorting matches numeric sorting."""
    if pd.api.types.is_numeric_dtype(s):
        nn = s.dropna()
        integral = bool(len(nn) == 0 or np.all(np.isclose(nn.values, np.round(nn.values))))
        if integral:
            width = max(3, len(str(int(nn.max()))) if len(nn) else 3)
            out = s.apply(lambda v: missing if pd.isna(v) else "{}_{}".format(name, str(int(v)).zfill(width)))
            return out.astype(str), True
    return s.fillna(missing).astype(str).str.strip(), False


dfx = df.copy()
print("--- building sortable labels ---")
for c in ["device", "site", "nurse"]:
    dfx[c], padded = to_sortable_label(df[c], c)
    print("  {:7s} zero-padded: {:5s} | sample: {}".format(
        c, str(padded), sorted(dfx[c].unique())[:3]))

dfx["recording_year"] = recording_year_num.apply(
    lambda v: "__MISSING__" if pd.isna(v) else "{}".format(int(v)))

mask_1to9 = dfx["strat_fold"] <= 9
mask_f10 = dfx["strat_fold"] == 10
print("\nrecords: folds 1-9 = {} | fold 10 = {} | total = {}".format(
    int(mask_1to9.sum()), int(mask_f10.sum()), len(dfx)))

CRAMER = {}

# --- 7.1-7.3: device / site / nurse x strat_fold (full dataset) ---
for col, key in [("device", "device_x_fold"), ("site", "site_x_fold"), ("nurse", "nurse_x_fold")]:
    fname = PREFIX + "crosstab_" + col + "_fold.csv"
    print("\n" + "-" * 100)
    print("--- " + col + " x strat_fold (counts, full dataset) ---")
    ct = pd.crosstab(dfx[col], dfx["strat_fold"]).sort_index()
    ct_full = ct.copy()
    ct_full["TOTAL"] = ct.sum(axis=1)
    print(ct_full.to_string())
    rowshare = ct.div(ct.sum(axis=1), axis=0)
    print("\nrow share (% of each " + col + " within each fold):")
    print((rowshare * 100).round(2).to_string())
    stats = cramers_v(ct)
    CRAMER[key] = stats
    print("\nCramer's V ({} x strat_fold): {:.4f}  | bias-corrected: {:.4f}  "
          "| chi2={:.2f}, dof={}, n={}, shape={}".format(
              col, stats["cramers_v"], stats["cramers_v_biascorrected"],
              stats["chi2"], stats["dof"], stats["n"], stats["shape"]))
    save_csv(ct_full, fname, index=True)
    save_csv(rowshare, fname.replace(".csv", "_rowshare_raw.csv"), index=True)

# --- 7.4: superclass prevalence within each device / site - folds 1-9 only ---
# Fold 10 (test) is kept intentionally blind: no metadata x label table is ever
# built on it.
lab_1to9 = lab.loc[mask_1to9]
overall_prev_1to9 = lab_1to9.mean()
PREV_1TO9 = {}
for col in ["device", "site"]:
    fname = PREFIX + "prevalence_" + col + "_superclass_folds1to9.csv"
    print("\n" + "-" * 100)
    print("--- superclass prevalence within each " + col + "  [folds 1-9 only; fold 10 blind] ---")
    joined = dfx.loc[mask_1to9, [col]].join(lab_1to9)
    prev = joined.groupby(col)[SUPERCLASSES].mean()
    prev.insert(0, "n_records", joined.groupby(col).size())
    prev = prev.sort_values("n_records", ascending=False)
    PREV_1TO9[col] = prev

    disp = prev.copy()
    for c in SUPERCLASSES:
        disp[c] = (disp[c] * 100).round(2)
    overall_row = pd.DataFrame(
        [[int(mask_1to9.sum())] + [round(float(overall_prev_1to9[c]) * 100, 2) for c in SUPERCLASSES]],
        index=["__OVERALL_folds1to9__"], columns=["n_records"] + SUPERCLASSES)
    disp = pd.concat([overall_row, disp])
    print("(figures in percent; the __OVERALL_folds1to9__ row is the baseline for this subset)")
    print(disp.to_string())

    print("\nmax deviation from the folds1-9 overall prevalence, in percentage points:")
    spread = (prev[SUPERCLASSES].sub(overall_prev_1to9, axis=1).abs() * 100)
    print(spread.max().round(2).to_frame("max_abs_deviation_pp").to_string())

    save_csv(disp, fname, index=True)
    save_csv(prev, fname.replace(".csv", "_raw.csv"), index=True)

# --- fold-10 marginal prevalence = AUPRC chance baseline (marginal only, no crosstab) ---
print("\n" + "-" * 100)
print("--- AUPRC chance baseline on fold 10 (marginal prevalence only) ---")
lab_f10 = lab.loc[mask_f10]
CHANCE_F10 = {"n_records_fold10": int(mask_f10.sum()), "prevalence": {}, "counts": {}}
rows = []
for c in SUPERCLASSES:
    cnt = int(lab_f10[c].sum())
    prv = float(lab_f10[c].mean()) if len(lab_f10) else float("nan")
    CHANCE_F10["counts"][c] = cnt
    CHANCE_F10["prevalence"][c] = prv
    rows.append({"superclass": c, "n_positive": cnt, "prevalence": round(prv, 6),
                 "prevalence_pct": round(prv * 100, 3)})
f10_tab = pd.DataFrame(rows).set_index("superclass")
print("n_records in fold 10:", int(mask_f10.sum()))
print(f10_tab.to_string())
print("\nReminder: chance AUPRC for each class = its prevalence. Chance macro-AUROC = 0.50.")
print("No device/site x label table was computed on fold 10 (intentional test blinding).")


# --- 7.5: unseen metadata categories across folds ---
print("\n" + "-" * 100)
print("--- 7.5 - unseen device / site / nurse categories across folds ---")
UNSEEN = {}
for col in ["device", "site", "nurse"]:
    sub = dfx[[col, "strat_fold"]]
    cats_train = set(sub.loc[sub["strat_fold"] <= 8, col].unique())
    cats_f9 = set(sub.loc[sub["strat_fold"] == 9, col].unique())
    cats_f10 = set(sub.loc[sub["strat_fold"] == 10, col].unique())

    n_in_f9 = sub[sub["strat_fold"] == 9][col].value_counts()
    n_in_f10 = sub[sub["strat_fold"] == 10][col].value_counts()
    n_overall = sub[col].value_counts()

    new9 = sorted(cats_f9 - cats_train)
    new10 = sorted(cats_f10 - cats_train)
    nfolds = sub.groupby(col)["strat_fold"].nunique()
    single = sorted(nfolds[nfolds == 1].index)

    UNSEEN[col] = {
        "n_categories_total": int(sub[col].nunique()),
        "n_categories_in_folds1to8": len(cats_train),
        "in_fold9_not_in_folds1to8": [
            {"category": c, "n_records_in_fold9": int(n_in_f9.get(c, 0))} for c in new9],
        "in_fold10_not_in_folds1to8": [
            {"category": c, "n_records_in_fold10": int(n_in_f10.get(c, 0))} for c in new10],
        "single_fold_only": [
            {"category": c,
             "fold": int(sub.loc[sub[col] == c, "strat_fold"].iloc[0]),
             "n_records": int(n_overall.get(c, 0))} for c in single],
    }

    print("\n--- " + col + " ---")
    print("unique categories: {} | in folds 1-8: {}".format(
        UNSEEN[col]["n_categories_total"], UNSEEN[col]["n_categories_in_folds1to8"]))
    print("in fold 9 but not in 1-8 ({}): {}".format(
        len(new9), UNSEEN[col]["in_fold9_not_in_folds1to8"] if new9 else "none"))
    print("in fold 10 but not in 1-8 ({}): {}".format(
        len(new10), UNSEEN[col]["in_fold10_not_in_folds1to8"] if new10 else "none"))
    print("only present in a single fold ({}): {}".format(
        len(single), UNSEEN[col]["single_fold_only"] if single else "none"))
    if new9 or new10:
        warn("{}: unseen categories exist in val/test - encoding must explicitly handle "
             "an unknown category or it will break or silently zero out at inference.".format(col))


# =====================================================================================
# SECTION 7.6 - TEMPORAL CONFOUNDING
# =====================================================================================
hdr("SECTION 7.6 - TEMPORAL CONFOUNDING")

for colA, colB, key, fname in [
    ("recording_year", "strat_fold", "recording_year_x_fold", PREFIX + "crosstab_recording_year_fold.csv"),
    ("device", "recording_year", "device_x_recording_year", PREFIX + "crosstab_device_recording_year.csv"),
]:
    print("\n" + "-" * 100)
    print("--- " + colA + " x " + colB + " (counts, full dataset) ---")
    ct = pd.crosstab(dfx[colA], dfx[colB]).sort_index()
    ct_full = ct.copy()
    ct_full["TOTAL"] = ct.sum(axis=1)
    print(ct_full.to_string())
    stats = cramers_v(ct)
    CRAMER[key] = stats
    print("\nCramer's V ({} x {}): {:.4f}  | bias-corrected: {:.4f}  "
          "| chi2={:.2f}, dof={}, n={}, shape={}".format(
              colA, colB, stats["cramers_v"], stats["cramers_v_biascorrected"],
              stats["chi2"], stats["dof"], stats["n"], stats["shape"]))
    save_csv(ct_full, fname, index=True)


# =====================================================================================
# SECTION 7.7 - PERMUTATION CHANCE BASELINE FOR METADATA-LABEL ASSOCIATION
# =====================================================================================
hdr("SECTION 7.7 - PERMUTATION CHANCE BASELINE FOR METADATA-LABEL ASSOCIATION")
print("N_PERM = {} | PERM_SEED = {} | restricted to strat_fold <= 9 (fold 10 stays blind)".format(
    N_PERM, PERM_SEED))

rng_perm = np.random.default_rng(PERM_SEED)


def crosstab_counts(codes, y, K):
    idx = codes * 2 + y
    return np.bincount(idx, minlength=K * 2).reshape(K, 2).astype(float)


def max_abs_dev_pp(codes, y, n_k, overall_prev):
    sums = np.bincount(codes, weights=y.astype(float), minlength=len(n_k))
    with np.errstate(divide="ignore", invalid="ignore"):
        cat_mean = np.where(n_k > 0, sums / np.where(n_k > 0, n_k, 1.0), np.nan)
    dev = np.abs(cat_mean - overall_prev) * 100
    return float(np.nanmax(dev))


ASSOC_RESULTS = {}
for col in ["device", "site", "nurse"]:
    codes, uniques = pd.factorize(dfx.loc[mask_1to9, col], sort=True)
    K = len(uniques)
    n_k = np.bincount(codes, minlength=K).astype(float)
    do_maxdev = col in ("device", "site")

    print("\n" + "-" * 100)
    print("--- " + col + " (K={} categories, n={} records in folds 1-9) ---".format(K, len(codes)))

    rows = []
    for sc in SUPERCLASSES:
        y = lab_1to9[sc].to_numpy().astype(np.int64)
        overall_prev = float(y.mean())

        obs_mat = crosstab_counts(codes, y, K)
        obs_v = cramers_v_arr(obs_mat)["cramers_v_biascorrected"]
        obs_maxdev = max_abs_dev_pp(codes, y, n_k, overall_prev) if do_maxdev else float("nan")

        null_v = np.empty(N_PERM)
        null_maxdev = np.empty(N_PERM) if do_maxdev else None
        for i in range(N_PERM):
            y_perm = rng_perm.permutation(y)
            null_v[i] = cramers_v_arr(crosstab_counts(codes, y_perm, K))["cramers_v_biascorrected"]
            if do_maxdev:
                null_maxdev[i] = max_abs_dev_pp(codes, y_perm, n_k, overall_prev)

        p_v = (1 + int(np.sum(null_v >= obs_v))) / (N_PERM + 1)
        row = {
            "superclass": sc,
            "v_observed": obs_v,
            "v_null_mean": float(null_v.mean()),
            "v_null_p95": float(np.percentile(null_v, 95)),
            "v_pvalue": float(p_v),
        }
        if do_maxdev:
            p_md = (1 + int(np.sum(null_maxdev >= obs_maxdev))) / (N_PERM + 1)
            row.update({
                "maxdev_pp_observed": obs_maxdev,
                "maxdev_pp_null_mean": float(null_maxdev.mean()),
                "maxdev_pp_null_p95": float(np.percentile(null_maxdev, 95)),
                "maxdev_pp_pvalue": float(p_md),
            })
        else:
            row.update({"maxdev_pp_observed": float("nan"), "maxdev_pp_null_mean": float("nan"),
                        "maxdev_pp_null_p95": float("nan"), "maxdev_pp_pvalue": float("nan")})
        rows.append(row)

    tab = pd.DataFrame(rows).set_index("superclass")
    print(tab.round(5).to_string())
    fname = PREFIX + "assoc_permutation_" + col + ".csv"
    save_csv(tab, fname, index=True)
    ASSOC_RESULTS[col] = tab.reset_index().to_dict(orient="records")

    for sc, r in zip(SUPERCLASSES, rows):
        if r["v_pvalue"] <= 0.05:
            note("{} x {}: observed bias-corrected V={:.4f} exceeds the permutation null "
                 "(p={:.4f}) - the association is unlikely to be chance.".format(
                     col, sc, r["v_observed"], r["v_pvalue"]))


# =====================================================================================
# SECTION 8 - MANIFEST
# =====================================================================================
hdr("SECTION 8 - MANIFEST")

manifest = {
    "code_version": CODE_VERSION,
    "run_started": RUN_STARTED,
    "run_finished": datetime.now().astimezone().isoformat(),
    "dataset_root": DATA_ROOT,
    "dataset_folder_name": os.path.basename(os.path.normpath(DATA_ROOT)),
    "output_dir": OUT_DIR,
    "input_files_sha256": INPUT_HASHES,
    "other_inputs_sha256": OTHER_HASHES,
    "sha256sums_verification": SHA256SUMS_VERIFICATION,
    "records_mismatch_diagnosis": RECORDS_MISMATCH_DIAGNOSIS,
    "library_versions": {
        "python": sys.version.replace("\n", " "),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "platform": platform.platform(),
    },
    "dataset_summary": {
        "n_records": int(len(df)),
        "n_patients": int(df["patient_id"].nunique()),
        "n_devices": int(df["device"].nunique(dropna=True)),
        "n_sites": int(df["site"].nunique(dropna=True)),
        "n_nurses": int(df["nurse"].nunique(dropna=True)),
        "superclass_counts": {c: int(lab[c].sum()) for c in SUPERCLASSES},
        "n_records_without_superclass": int((n_labels == 0).sum()),
        "n_age_sentinel_ge120": n_sentinel,
    },
    "fold_tab": FOLD_TAB_JSON,
    "metadata_patient_entanglement": METADATA_PATIENT_ENTANGLEMENT,
    "metadata_concentration": METADATA_CONCENTRATION,
    "label_convention": LABEL_CONVENTION,
    "label_likelihood_sensitivity": {
        "alt_rule": "keep only codes with likelihood > 0",
        "n_changed": n_changed,
        "pct_changed": round(n_changed / len(df) * 100, 4),
        "superclass_counts_primary": {c: int(lab[c].sum()) for c in SUPERCLASSES},
        "superclass_counts_alt": alt_counts,
    },
    "cramers_v": CRAMER,
    "auprc_chance_baseline_fold10": CHANCE_F10,
    "unseen_categories": UNSEEN,
    "metadata_label_association_permutation": {
        "N_PERM": N_PERM,
        "PERM_SEED": PERM_SEED,
        "restricted_to": "strat_fold <= 9",
        "results": ASSOC_RESULTS,
    },
    "checks": {
        "patient_disjoint_folds": "FAIL" if LEAKAGE_FAIL else "PASS",
        "n_patients_in_multiple_folds": int(len(violators)),
        "records_manifest_mismatch_csv_only": len(only_csv),
        "records_manifest_mismatch_records_only": len(only_rec),
    },
    "warnings": WARNINGS,
    "outputs": [],
}

for name in OUTPUT_FILES:
    p = os.path.join(OUT_DIR, name)
    manifest["outputs"].append({"file": name, "bytes": os.path.getsize(p), "sha256": sha256_of(p)})

MANIFEST_PATH = os.path.join(OUT_DIR, PREFIX + "manifest.json")
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, default=_json_default)

print("[saved] " + MANIFEST_PATH)
print("\n--- manifest top-level keys (console shows sizes only; the file on disk is complete) ---")
for k, v in manifest.items():
    size_chars = len(json.dumps(v, ensure_ascii=False, default=_json_default))
    print("  {:42s} size={} chars".format(k, size_chars))
print("\nmanifest path  :", MANIFEST_PATH)
print("manifest sha256:", sha256_of(MANIFEST_PATH))


# =====================================================================================
# FINAL SUMMARY (for the stage report)
# =====================================================================================
hdr("SUMMARY - " + CODE_VERSION)
print("records / patients        :", len(df), "/", df["patient_id"].nunique())
print("device / site / nurse     :", df["device"].nunique(dropna=True), "/",
      df["site"].nunique(dropna=True), "/", df["nurse"].nunique(dropna=True))
print("checksum verification     :", {k: (v["status"] if isinstance(v, dict) and "status" in v else v)
                                       for k, v in SHA256SUMS_VERIFICATION.items()})
print("RECORDS mismatch          : explained={} unexplained={}".format(
    len(RECORDS_MISMATCH_DIAGNOSIS["explained_lines"]), len(RECORDS_MISMATCH_DIAGNOSIS["unexplained_lines"])))
print("age sentinel (>=120)      :", n_sentinel)
print("no superclass             :", int((n_labels == 0).sum()))
print("multi-label (>=2)         :", int((n_labels >= 2).sum()))
print("patient-disjoint folds    :", "FAIL" if LEAKAGE_FAIL else "PASS")
for key, colname in [("device_x_fold", "device"), ("site_x_fold", "site"), ("nurse_x_fold", "nurse"),
                     ("recording_year_x_fold", "year x fold"), ("device_x_recording_year", "device x year")]:
    print("Cramer's V {:14s} : {:.4f} (corrected {:.4f})".format(
        colname, CRAMER[key]["cramers_v"], CRAMER[key]["cramers_v_biascorrected"]))
print("AUPRC chance (fold 10)    :",
      {c: round(CHANCE_F10["prevalence"][c], 4) for c in SUPERCLASSES})
print("unseen categories in val/test:",
      {c: len(UNSEEN[c]["in_fold9_not_in_folds1to8"]) + len(UNSEEN[c]["in_fold10_not_in_folds1to8"])
       for c in ["device", "site", "nurse"]})
print("effective cardinality     :",
      {c: METADATA_CONCENTRATION[c]["effective_n_categories"] for c in ["device", "site", "nurse"]})
print("files produced            :", len(OUTPUT_FILES) + 1)
print("\nwarnings ({}):".format(len(WARNINGS)))
for w in WARNINGS:
    print("  - " + w)

if LEAKAGE_FAIL:
    raise AssertionError(
        "Patient-level leakage: {} patients appear in more than one strat_fold. "
        "Audit outputs were written; resolve this before any modeling.".format(len(violators)))

print("\n" + CODE_VERSION + " finished successfully.")


SECTION 1 - PROVENANCE
run started         : 2026-10-02T03:35:48.860252+03:30
code version        : v0.1.2
python              : 3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]
platform            : Windows-10-10.0.19045-SP0
machine / processor : AMD64 | Intel64 Family 6 Model 186 Stepping 2, GenuineIntel
numpy               : 1.26.4
pandas              : 2.3.3
dataset folder name : ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3
dataset root        : D:\ptbxl-metadata-audit\data\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3
output dir          : D:\ptbxl-metadata-audit\outputs\v0.1.2

--- SHA-256 of the main input files (computed) ---
ptbxl_database.csv         7600de9c1b27d181d850b3c6038a35d7c3ddb6bb33b702e3a20252a6859d216b   (6,594,879 bytes)
scp_statements.csv         ad05b0b1fcae83bb1230755ad9cfc7c96f303feddc08a4a9ad5bdc9ca63bac8f   (9,720 bytes)

--- looking for a checksum file ---
checksu